In [94]:
import pandas as pd

In [95]:
data_path = "../player-data/Master.csv"
df = pd.read_csv(data_path)

In [96]:
print(df.columns)

Index(['playerID', 'birthYear', 'birthMonth', 'birthDay', 'birthCountry',
       'birthState', 'birthCity', 'deathYear', 'deathMonth', 'deathDay',
       'deathCountry', 'deathState', 'deathCity', 'nameFirst', 'nameLast',
       'nameGiven', 'weight', 'height', 'bats', 'throws', 'debut', 'finalGame',
       'retroID', 'bbrefID'],
      dtype='object')


In [ ]:
multiples_count = (df['nameGiven'].value_counts() > 1).sum()
print(multiples_count)

2125


In [82]:
df_unique = df[['playerID', 'nameGiven', 'nameFirst', 'nameLast']].copy()
df_unique

,playerID,nameGiven,nameFirst,nameLast
0,aardsda01,David Allan,David,Aardsma
1,aaronha01,Henry Louis,Hank,Aaron
2,aaronto01,Tommie Lee,Tommie,Aaron
3,aasedo01,Donald William,Don,Aase
4,abadan01,Fausto Andres,Andy,Abad
...,...,...,...,...
18841,zupofr01,Frank Joseph,Frank,Zupo
18842,zuvelpa01,Paul,Paul,Zuvella
18843,zuverge01,George,George,Zuverink
18844,zwilldu01,Edward Harrison,Dutch,Zwilling


In [83]:
# check for null values in given name column
df_unique['nameGiven'].isna().sum()

np.int64(39)

In [84]:
df_unique['nameFirst'].isna().sum()

np.int64(39)

In [85]:
df_unique['nameLast'].isna().sum()

np.int64(0)

In [86]:
# all missing given names also have missing first names - delete these rows
df_unique[df_unique['nameFirst'].isna() & df_unique['nameGiven'].isna()].shape

(39, 4)

In [87]:
df_unique.dropna(subset=['nameGiven'], inplace=True)
df_unique.shape

(18807, 4)

In [88]:
mask = df_unique['nameGiven'].apply(lambda x: ' ' not in x)
names_to_fix = df_unique.loc[mask, 'nameGiven'].to_dict()
print(len(names_to_fix))

1633


In [89]:
bad_rows = []

for idx, name in names_to_fix.items():
    curr_row = df_unique.loc[idx]
    first = curr_row['nameFirst']
    last = curr_row['nameLast']

    if name == first:
        df_unique.at[idx, 'nameGiven'] = f"{first} {last}" # add last name to given name
    elif name == last:
        df_unique.at[idx, 'nameGiven'] = f"{first} {last}" # add first name to given name
    else:
        bad_rows.append(idx) # needs manual fixing
    
print(len(bad_rows))

759


In [90]:
df_unique = df_unique.drop(index=bad_rows)
df_unique.shape

(18048, 4)

In [91]:
df_unique['nameGiven'].value_counts()

nameGiven
John Joseph       74
William Henry     53
William Joseph    48
Michael Joseph    42
John William      40
                  ..
Mauro Paul         1
Billy Cordell      1
Rodney Blaine      1
Alfons Francis     1
Anthony Aaron      1
Name: count, Length: 12634, dtype: int64

In [107]:
name_counts = df_unique['nameGiven'].value_counts()

# Split into two DataFrames
df_duplicates = df_unique[df_unique['nameGiven'].isin(name_counts[name_counts > 1].index)]
df_nonduplicates = df_unique[df_unique['nameGiven'].isin(name_counts[name_counts == 1].index)]

print("Duplicates:", len(df_duplicates))
print("Non-duplicates:", len(df_nonduplicates))

Duplicates: 7326
Non-duplicates: 10722


In [ ]:
players = (
    df_nonduplicates[['nameGiven']]
    .sort_values('nameGiven')
    .reset_index(drop=True)
)

# players.to_csv('players_nonduplicates.csv', index=False)
# print(f"{len(players)} unique players written to players_nonduplicates.csv")

players.iloc[:2000].to_csv('players_batchA.csv', index=False)
players.iloc[2000:4000].to_csv('players_batchB.csv', index=False)
players.iloc[4000:7000].to_csv('players_batchC.csv', index=False)
players.iloc[7000:].to_csv('players_batchD.csv', index=False)


10722 unique players written to players_nonduplicates.csv
